<a href="https://colab.research.google.com/github/123ufogit/GoogleAntigravity/blob/main/FisheyeBinaryHistConverter_Drive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Google Drive 連携 & 入出力フォルダ選択 UI


In [ ]:
# ============================================
# ① Google Drive マウント & 入力フォルダ選択 UI
# ============================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import ipywidgets as widgets
from IPython.display import display

ROOT = "/content/drive/MyDrive"

# 再帰的にフォルダ一覧を取得
def list_folders_recursive(root):
    folder_list = []
    for current_root, dirs, files in os.walk(root):
        for d in dirs:
            full_path = os.path.join(current_root, d)
            display_name = full_path.replace(root, "").lstrip("/")
            folder_list.append((display_name, full_path))
    return folder_list

folder_options = list_folders_recursive(ROOT)

input_dropdown = widgets.Dropdown(
    options=folder_options,
    description='入力フォルダ:',
    layout=widgets.Layout(width='500px')
)

button = widgets.Button(
    description='フォルダを確定',
    button_style='success'
)

selected_input = None

def on_button_clicked(b):
    global selected_input
    selected_input = input_dropdown.value
    print("入力フォルダ:", selected_input)

button.on_click(on_button_clicked)

display(input_dropdown, button)

Mounted at /content/drive


Dropdown(description='入力フォルダ:', layout=Layout(width='500px'), options=(('令和3年度', '/content/drive/MyDrive/令和3年度…

Button(button_style='success', description='フォルダを確定', style=ButtonStyle())

# Fisheye変換関数



In [ ]:
import math
import numpy as np
from PIL import Image

def equirectangular_to_fisheye_equidistant(
    src_img: Image,
    out_size: int = 1024,
    fov_deg: float = 180.0,
    yaw_deg: float = 0.0
) -> Image:

    W, H = src_img.size
    src_np = np.array(src_img.convert('RGB'), dtype=np.uint8)

    R = out_size // 2
    cx, cy = R, R
    dst = np.zeros((out_size, out_size, 3), dtype=np.uint8)

    # FOV → 最大角度
    fov_rad = math.radians(fov_deg)
    theta_max = fov_rad / 2.0

    # equidistant: r = f * θ → f = R / θ_max
    f = R / theta_max

    yaw_rad = math.radians(yaw_deg)

    def sample_equirectangular(lon, lat):
        u = (lon + math.pi) / (2 * math.pi) * W
        v = (math.pi/2 - lat) / math.pi * H

        # 境界値チェックを追加 (0 ～ W-1 / 0 ～ H-1 に収める)
        u0 = int(u) % W
        u1 = (u0 + 1) % W
        v0 = max(0, min(H - 1, int(v)))
        v1 = max(0, min(H - 1, v0 + 1))

        fu = u - int(u)
        fv = v - int(v)

        p00 = src_np[v0, u0]
        p01 = src_np[v0, u1]
        p10 = src_np[v1, u0]
        p11 = src_np[v1, u1]

        top = (1 - fu) * p00 + fu * p01
        bottom = (1 - fu) * p10 + fu * p11
        return ((1 - fv) * top + fv * bottom).astype(np.uint8)

    for y in range(out_size):
        for x in range(out_size):
            dx = x - cx
            dy = y - cy
            r = math.sqrt(dx*dx + dy*dy)

            if r > R:
                continue

            # equidistant inverse projection
            theta = r / f
            phi = math.atan2(-dy, dx) + yaw_rad

            # 球面座標へ (角度を -PI/2 ~ PI/2 の範囲に制限)
            lat = math.asin(max(-1.0, min(1.0, math.cos(theta))))
            lon = phi

            dst[y, x] = sample_equirectangular(lon, lat)

    return Image.fromarray(dst)

In [ ]:
# ============================================
# ② Fisheye（Orthographic）変換関数
# ============================================
import math
import numpy as np
from PIL import Image

def equirectangular_to_fisheye(
    src_img: Image,
    out_size: int = 1024,
    fov_deg: float = 180.0,
    yaw_deg: float = 0.0
) -> Image:

    W, H = src_img.size
    src_np = np.array(src_img.convert('RGB'), dtype=np.uint8)

    R = out_size // 2
    cx, cy = R, R
    dst = np.zeros((out_size, out_size, 3), dtype=np.uint8)

    fov_rad = math.radians(fov_deg)
    theta_max = fov_rad / 2.0
    yaw_rad = math.radians(yaw_deg)
    sin_tmax = math.sin(theta_max)

    def rnorm_to_theta(rn: float) -> float:
        val = rn * sin_tmax
        val = max(0.0, min(1.0, val))
        return math.asin(val)

    def sample_equirectangular(lon: float, lat: float) -> np.ndarray:
        u = (lon + math.pi) / (2.0 * math.pi) * W
        v = (math.pi / 2.0 - lat) / (math.pi) * H
        v = max(0.0, min(H - 1, v))

        u0 = int(u) % W
        u1 = (u0 + 1) % W
        v0 = int(v)
        v1 = min(H - 1, v0 + 1)

        fu = u - int(u)
        fv = v - int(v)

        p00 = src_np[v0, u0]
        p01 = src_np[v0, u1]
        p10 = src_np[v1, u0]
        p11 = src_np[v1, u1]

        interp_top = (1 - fu) * p00 + fu * p01
        interp_bottom = (1 - fu) * p10 + fu * p11
        pixel = (1 - fv) * interp_top + fv * interp_bottom

        return pixel.astype(np.uint8)

    for y in range(out_size):
        for x in range(out_size):
            dx, dy = x - cx, y - cy
            r = math.sqrt(dx*dx + dy*dy)

            if r <= R:
                rn = r / R
                theta = rnorm_to_theta(rn)
                phi = math.atan2(dy, dx) + yaw_rad

                lon = phi
                lat = math.pi / 2.0 - theta

                dst[y, x] = sample_equirectangular(lon, lat)

    return Image.fromarray(dst)

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import sys

# 前のセルで定義された関数を確認
# sJ9MZWvBGAMp: equirectangular_to_fisheye_equidistant
# 0bDmQFbE61bb: equirectangular_to_fisheye

try:
    # 関数の参照を保持
    def_equirectangular_to_fisheye_orthographic = equirectangular_to_fisheye
    def_equirectangular_to_fisheye_equidistant = equirectangular_to_fisheye_equidistant

    transform_type_dropdown = widgets.Dropdown(
        options=[('Orthographic', 'Orthographic'), ('Equidistant', 'Equidistant')],
        value='Equidistant',  # デフォルト値を Equidistant に変更
        description='変換タイプ:',
        layout=widgets.Layout(width='300px')
    )

    confirm_transform_button = widgets.Button(
        description='変換タイプを確定',
        button_style='info'
    )

    def on_confirm_transform_button_clicked(b):
        global equirectangular_to_fisheye
        selected_type = transform_type_dropdown.value

        if selected_type == 'Orthographic':
            equirectangular_to_fisheye = def_equirectangular_to_fisheye_orthographic
            print("使用する Fisheye 変換タイプ: Orthographic")
        elif selected_type == 'Equidistant':
            equirectangular_to_fisheye = def_equirectangular_to_fisheye_equidistant
            print("使用する Fisheye 変換タイプ: Equidistant")

    confirm_transform_button.on_click(on_confirm_transform_button_clicked)
    display(transform_type_dropdown, confirm_transform_button)

    # 初期設定を Equidistant に変更
    equirectangular_to_fisheye = def_equirectangular_to_fisheye_equidistant
    print("初期設定完了: Equidistant が選択されています。")

except NameError as e:
    print(f"⚠️ エラー: 関数が定義されていません。\n上の2つの変換関数セル（sJ9MZWvBGAMp と 0bDmQFbE61bb）を先に実行してください。")
    print(f"詳細: {e}")

Dropdown(description='変換タイプ:', index=1, layout=Layout(width='300px'), options=(('Orthographic', 'Orthographic'…

Button(button_style='info', description='変換タイプを確定', style=ButtonStyle())

初期設定完了: Equidistant が選択されています。


# 画像一括変換の実行

In [ ]:
# --- 閾値プルダウン ---
import ipywidgets as widgets
from IPython.display import display

threshold_dropdown = widgets.Dropdown(
    options=[160, 170, 180, 190, 200],
    value=180,
    description='閾値:',
    layout=widgets.Layout(width='200px')
)

display(threshold_dropdown)


Dropdown(description='閾値:', index=2, layout=Layout(width='200px'), options=(160, 170, 180, 190, 200), value=18…

In [ ]:
# ============================================
# ③ fisheye → 二値化 → ヒストグラム → CSV 保存
# ============================================

import cv2
import matplotlib.pyplot as plt
from datetime import datetime
import glob
import os
import numpy as np
from PIL import Image

if selected_input is None:
    print("⚠️ まず入力フォルダを選択して『フォルダを確定』を押してください。")
else:
    # 入力フォルダ名
    input_basename = os.path.basename(selected_input)

    # OUTPUT フォルダ生成
    output_root = os.path.join(selected_input, f"{input_basename}_OUTPUT")
    fisheye_dir = os.path.join(output_root, "fisheye")
    binary_dir  = os.path.join(output_root, "binary")
    hist_dir    = os.path.join(output_root, "hist")
    csv_dir     = os.path.join(output_root, "csv")

    for d in [output_root, fisheye_dir, binary_dir, hist_dir, csv_dir]:
        os.makedirs(d, exist_ok=True)

    print("出力フォルダ:", output_root)

    # 入力画像一覧
    image_files = []
    for ext in ('*.jpg', '*.jpeg', '*.png'):
        image_files.extend(glob.glob(os.path.join(selected_input, ext)))

    print(f"処理対象画像: {len(image_files)} 枚")

    # CSV 用データ
    count_results = []
    threshold_value = threshold_dropdown.value
    resize_size = 1024

    for i, img_path in enumerate(image_files):
        fname = os.path.basename(img_path)
        print(f"[{i+1}/{len(image_files)}] {fname} を処理中…")

        # -----------------------------
        # ① fisheye 変換 (fov_deg=360.0 を指定)
        # -----------------------------
        with Image.open(img_path) as src_image:
            fisheye_img = equirectangular_to_fisheye(src_image, fov_deg=360.0)

        fisheye_path = os.path.join(fisheye_dir, fname.replace(".", "_fisheye."))
        fisheye_img.save(fisheye_path)

        # -----------------------------
        # ② 二値化処理
        # -----------------------------
        img_gray = cv2.cvtColor(np.array(fisheye_img), cv2.COLOR_RGB2GRAY)
        _, binary = cv2.threshold(img_gray, threshold_value, 255, cv2.THRESH_BINARY)

        binary_resized = cv2.resize(binary, (resize_size, resize_size), interpolation=cv2.INTER_NEAREST)

        # 円形マスク
        h, w = binary_resized.shape
        cx, cy = w // 2, h // 2
        r = min(cx, cy)

        mask = np.zeros_like(binary_resized, dtype=np.uint8)
        cv2.circle(mask, (cx, cy), r, 1, -1)
        inside = (mask == 1)

        white_count = np.sum((binary_resized == 255) & inside)
        black_count = np.sum((binary_resized == 0) & inside)
        gap_fraction = white_count / (white_count + black_count) * 100

        # 二値化画像保存
        bin_path = os.path.join(binary_dir, fname.replace(".", "_bin_1024."))
        cv2.imwrite(bin_path, binary_resized)

        # -----------------------------
        # ③ ヒストグラム
        # -----------------------------
        plt.figure(figsize=(6,4))
        plt.hist(img_gray.ravel(), bins=256, range=(0,256), color='gray')
        plt.title(f"Histogram: {fname}")
        plt.axvline(threshold_value, color='red', linestyle='--')
        plt.ylim(0, 200000)
        hist_path = os.path.join(hist_dir, fname.replace(".", "_hist."))
        plt.savefig(hist_path)
        plt.close()

        # CSV データ追加
        count_results.append([fname, white_count, black_count, gap_fraction])

    # -----------------------------
    # ④ CSV 保存
    # -----------------------------
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_path = os.path.join(csv_dir, f"pixel_count_{timestamp}.csv")

    with open(csv_path, "w", encoding="utf-8") as f:
        f.write("filename,white_pixels,black_pixels,gap_fraction_percent\n")
        for row in count_results:
            f.write(f"{row[0]},{row[1]},{row[2]},{row[3]:.2f}\n")

import IPython.display as ipd
ipd.display(ipd.Javascript('new Audio("https://www.soundjay.com/misc/sounds/bell-ringing-05.mp3").play()'))

from IPython.display import Javascript

Javascript("""
Notification.requestPermission().then(function(result) {
  if (result === 'granted') {
    new Notification('処理が完了しました！');
  }
});
""")
from IPython.display import HTML

folder_name = os.path.basename(output_root)
search_url = f"https://drive.google.com/drive/search?q={folder_name}"

print("\n🎉 すべての処理が完了しました！")
print("出力フォルダ:", output_root)
display(HTML(f'<a href="{search_url}" target="_blank">OUTPUT フォルダを開く</a>'))

出力フォルダ: /content/drive/MyDrive/Colab Notebooks/Add_LatLon/TEST/TEST_OUTPUT
処理対象画像: 12 枚
[1/12] 100_cam_1st_xmp_e.jpg を処理中…
[2/12] 101_cam_2nd_xmp_e.jpg を処理中…
[3/12] 100_cam_2nd_xmp_e.jpg を処理中…
[4/12] 101_cam_1st_xmp_e.jpg を処理中…
[5/12] 103_cam_2nd_xmp_e.jpg を処理中…
[6/12] 102_cam_2nd_xmp_e.jpg を処理中…
[7/12] 103_cam_1st_xmp_e.jpg を処理中…
[8/12] 102_cam_1st_xmp_e.jpg を処理中…
[9/12] 104_cam_1st_xmp_e.jpg を処理中…
[10/12] 104_cam_2nd_xmp_e.jpg を処理中…
[11/12] 105_cam_1st_xmp_e.jpg を処理中…
[12/12] 105_cam_2nd_xmp_e.jpg を処理中…


<IPython.core.display.Javascript object>


🎉 すべての処理が完了しました！
出力フォルダ: /content/drive/MyDrive/Colab Notebooks/Add_LatLon/TEST/TEST_OUTPUT
